# **MyGPT**
Let's build a simple GPT Language Model for a text generation task.<br>
I'll train it on a Quiet Dan dataset (because I can)

In [1]:
from datasets import load_dataset
import torch
from torch import nn, optim
from torch.nn import functional as F
import tqdm

ds = load_dataset("mahiatlinux/TinyStories-GPT4-V2-50K-SUBSET")

c:\Users\rayga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

### **Data Setup**

In [3]:
data = "".join(list(ds["train"]["text"])[:5_000])
vocab = sorted(set(data))
vocab_size = len(vocab)

itos = {i:s for i, s in enumerate(vocab)}
stoi = {s:i for i, s in itos.items()}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join([itos[i] for i in l])

In [4]:
encode("Once upon a time,"), decode(encode("Once upon a time,"))

([32, 57, 46, 48, 1, 64, 59, 58, 57, 1, 44, 1, 63, 52, 56, 48, 5],
 'Once upon a time,')

In [5]:
data = torch.tensor(encode(data))

In [6]:
data.shape

torch.Size([4004439])

In [7]:
ratio = int(len(data) * 0.9)
train_data = data[:ratio]
val_data = data[ratio:]

In [8]:
batch_size = 64
block_size = 256
dropout = 0.2
emb_dim = 384
n_heads = 6
n_blocks = 4

In [9]:
data[:5]

tensor([32, 57, 46, 48,  1])

In [10]:
def get_batch(subset="train"):
    if subset == "train":
        dataset = train_data
    else:
        dataset = val_data
    idcs = torch.randint(len(dataset) - block_size, (batch_size,))
    xb = torch.stack([data[ix:ix+block_size] for ix in idcs])
    yb = torch.stack([data[ix+1:ix+block_size+1] for ix in idcs])
    return xb, yb

In [11]:
xb, yb = get_batch("train")
xb.shape, yb.shape

(torch.Size([64, 256]), torch.Size([64, 256]))

In [12]:
xb[0, :10], yb[0, :10]

(tensor([57, 44, 56, 48, 47,  1, 37, 52, 56,  7]),
 tensor([44, 56, 48, 47,  1, 37, 52, 56,  7,  1]))

### **GPT itself!**

In [46]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):
        q = self.query(xb)
        k = self.key(xb)
        v = self.value(xb)
        B, T, C = q.shape  # head_size

        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T) table
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)  # (B, T, T)
        out = wei @ v  # (B, T, T) @ (B, T, head_size)
        return out

In [47]:
class MultiHead(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, xb):
        out = torch.cat([head(xb) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [48]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.ffwd = nn.Sequential(
            nn.Linear(emb_dim, emb_dim*4),
            nn.ReLU(),
            nn.Linear(emb_dim*4, emb_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ffwd(xb)

In [49]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHead(n_heads, emb_dim//n_heads)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)
    
    def forward(self, xb):
        xb = xb + self.sa(self.ln1(xb))
        out = xb + self.ffwd(self.ln2(xb))
        return out

In [50]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.chr_enc = nn.Embedding(vocab_size, emb_dim)
        self.pos_enc = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_blocks)])
        self.ln = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)
    
    def forward(self, input, target=None):
        B, T = input.shape
        # input shape: (B, T)
        input = self.chr_enc(input)  # (B, T, C)
        input += self.pos_enc(torch.arange(T, device=device))
        logits = self.blocks(input)
        logits = self.lm_head(self.ln(logits))
        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
        return logits, loss
    
    def generate(self, idx, max_tokens):
        # idx is a (B, T) shape int tensor
        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]  # Sliding over a big context
            logits, loss = self(idx_cond)
            # Logits are of shape (B, T, C)
            logits = logits[:, -1, :]  # Last character pick  (B, C)
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.concat((idx, next_token), dim=1)  # (B, T+1, C)
        return idx

In [51]:
model = GPT().to(device)
logits, loss = model(xb.to(device), yb.to(device))

In [53]:
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_tokens=100)[0].tolist()))


XbVs.Vn“cgfAx…–u"oa3JD;,WG8ACQ-”P:dmI?KAquS?3T!K1T'oN—p'hVdqBDXuoxp‘
;z/?lh–8'LIrAPZmcdIk”i-l?h:—gbt


In [ ]:
"""
Optimization Loop
"""
optimizer = optim.Adam(model.parameters(), lr=3e-4)
for epoch in tqdm.tqdm(range(6_000)):
    xb, yb = get_batch("train")
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(loss)

100%|██████████| 6000/6000 [20:01<00:00,  4.99it/s]


tensor(0.8072, device='cuda:0', grad_fn=<NllLossBackward0>)


In [55]:
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_tokens=1000)[0].tolist()))


Mia was glad but she knew Sarah was curious that Tom and dug. They decided to buy the kitch and have some bright go than her. Jill had said, "Yuck, you cont, but you cush don't with some." So Jill started to sang the subway bite for her wagone! 
When they went back to Bill got and looked, and Tim's mom started to carry. Whisked her fact act helped the sunflower back to the beach. Tim was not broken anywager.
Every day, Tim's mom painted Emma. So, he decided soon they were sad because he knew people. Suddenly, I'll be groner the vaset out dizzy. So, he felt so carefully. The next day, the bird came to Pete. Tim and Suddy heard they had a leaf to the sight. All the other mom heard the story. They had so much fun together.
As they walked to look at the believes and warmed and saw the car and they both opened its friends. Fish waved growled the woodien of them. All fish was full and flew down, his friends, and he was tiny, grabbig the car and the little hair and he watched his water in th

# **Having a Lot of fun**

In [58]:
@torch.inference_mode()
def nabalabobit(prompt):
    prompt = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    return decode(model.generate(prompt, max_tokens=1000)[0].tolist())

In [61]:
print(nabalabobit("One bird said to another:"))

One bird said to another:. The bird lived in angry, something at the bird. Tim liked to play with it.
"Sure, a big, a big bird!" said.
"Tim, that Jim had a tiny wind a wide of sleep and a bright. "I can't do it!" 
Jink then saw Ane and started it. He started to sit on. "See, you don't know, it is okay. We can try to his friends for the answe pet from the friend."
His dad mom smiled and said, "Yes, you are a good friends field. Do you want to go right to go soft and say. They got home and finally."
Ben and Ben say, Emily and Dad sorry. They ran to the hill. They ate celery.
"Look, Mom, why are so pretends we go. How want to be clean up the music on them."
Mom she even again. He used the musican. He opens Ben and Sara took his feel sad. They has dad. They learned a lemry friend. He learned that it is a march and leaf. The musicial sticks could take all over a ball.
Then, Sure asked if he turned it would make a nice on a march. They ran to the tree and hugged her. Sue was so happy and scr

In [65]:
print(nabalabobit("Girl in white was sitting on the sofa with 5 men behind."))

Girl in white was sitting on the sofa with 5 men behind. The sun was very delight! Everyone was so spill and they could eat the sheet day and lone all night while was. 
Then, it thin the sun came to Tim slapped. At the end of the show was tutor. Tim learned that he showed the cage but it hurt he could not hop. Then, he played in the sun. They went home, Bob saw a lemovie of countain. He thout he worked and went outside.
He knew it was to become his toy and leave. She ran to Can Mia to help him the sun. They stopped some in its mix.
"Come on this count," Bob said. "Maybe it will bark above it. Before," Lily said, says. She looks around with her bags on the hosk green of the sun counters. "You use a big tree. That are your are, it's bite!"
"Yes, Mommy. I told your toys, let's play again."
"I will eat it," Ben says. They do.
Theysee happy. They like the toys.
Mad laiging aigh flowers. She gives the tree, and Mummy in the big nough across! She only to dea les Molly last them. Mommy seees w

In [63]:
print(nabalabobit("Men in black met an alien."))

Men in black met an alien. They hug tail and ate a big page. They also lended and smiles.
"Oh, this is this owner. And this. I need sudder the hcle," ans says.
"OK, I will polish. I can make a caselet. Let's get pole with in the shelf."
Anna and Ben came out close. They work inside their shapes and try. Len is that can't fun?" The Amy wants the ant, then she wants held. They rup the room.
"I'm proud of your pints. I read that. This my can make some ends him. She asks. They hug a bag with the ancient from playing with the cat. They run and fold each other from that day.
But they more was making a vale soft, a friendly rock. The town were clets having its beak clear and flew. They each other in, and they all at each others ofter broke, greeds. They hoped the store and keep their mom. They gave it the little tree and played alone.
As they went walking, they saw a big cat near a big dog. The dog. They wanted to teach and tasted to give the cat. They were sad because they learned that to be

## **Saving The Model**

In [66]:
torch.save(model.state_dict(), "miniGPT.pth")

In [67]:
m2 = GPT().to(device)
m2.load_state_dict(torch.load('miniGPT.pth', weights_only=True))

<All keys matched successfully>

In [71]:
print(decode(m2.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_tokens=100)[0].tolist()))


Sue and Ben and Lucy had a lot of big. She took Sue's work hard not answer. She choses the big dog c


In [75]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

7251533

## **Conclusion**
- Is it any good? 💯! It's better than LSTMs I built before, keep in mind this is a small model working on the level of characters.
- Tokenization improvement and model complexity increase is a natural continuation!

**Coolness Rate: Over The Fucking Top!%**

<img src="https://static.wikia.nocookie.net/cyberpunk/images/d/de/Johnny_Silverhand_Database_CP2077.png/revision/latest?cb=20231003222545" width=10%>